In [10]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

df = pd.read_csv("train.csv")


def section(title):
    display(Markdown(f"## {title}"))


section("Dataset overview")

overview = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Total cells",
        "Memory usage MB"
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        df.shape[0] * df.shape[1],
        round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    ]
})

display(overview)


section("Column types")

dtype_summary = df.dtypes.value_counts().reset_index()
dtype_summary.columns = ["dtype", "count"]

display(dtype_summary)

column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notnull().sum().values,
    "null_count": df.isnull().sum().values,
    "missing_%": (df.isnull().mean().values * 100),
    "unique_count": df.nunique(dropna=True).values,
    "unique_%": (df.nunique(dropna=True).values / len(df) * 100)
})

display(column_summary)


section("Missing values")

missing_df = column_summary[
    ["column", "null_count", "missing_%"]
].sort_values("missing_%", ascending=False)

missing_overview = pd.DataFrame({
    "metric": [
        "Total missing cells",
        "Total missing %",
        "Rows with at least one missing value",
        "Rows with at least one missing value %"
    ],
    "value": [
        df.isnull().sum().sum(),
        round(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2),
        df.isnull().any(axis=1).sum(),
        round(df.isnull().any(axis=1).mean() * 100, 2)
    ]
})

display(missing_overview)
display(missing_df)


section("Columns grouped by missingness")

missing_groups = pd.DataFrame({
    "group": [
        "More than 90% missing",
        "More than 50% missing",
        "Between 10% and 50% missing",
        "No missing values"
    ],
    "columns": [
        missing_df[missing_df["missing_%"] > 90]["column"].tolist(),
        missing_df[missing_df["missing_%"] > 50]["column"].tolist(),
        missing_df[(missing_df["missing_%"] >= 10) & (missing_df["missing_%"] <= 50)]["column"].tolist(),
        missing_df[missing_df["null_count"] == 0]["column"].tolist()
    ]
})

display(missing_groups)


section("Empty strings and whitespace")

object_cols = df.select_dtypes(include="object").columns

empty_string_info = []

for col in object_cols:
    empty_strings = (df[col] == "").sum()
    empty_or_whitespace = df[col].astype(str).str.strip().eq("").sum()

    if empty_strings > 0 or empty_or_whitespace > 0:
        empty_string_info.append({
            "column": col,
            "empty_strings": empty_strings,
            "empty_or_whitespace": empty_or_whitespace
        })

empty_string_df = pd.DataFrame(empty_string_info)

if len(empty_string_df) > 0:
    display(empty_string_df)
else:
    display(pd.DataFrame({"result": ["No empty or whitespace-only strings detected"]}))


section("Duplicates")

duplicates_df = pd.DataFrame({
    "check": [
        "Fully duplicated rows",
        "Duplicated Unique Key values" if "Unique Key" in df.columns else "Duplicated Unique Key values"
    ],
    "count": [
        df.duplicated().sum(),
        df["Unique Key"].duplicated().sum() if "Unique Key" in df.columns else np.nan
    ]
})

display(duplicates_df)


section("Cardinality analysis")

cardinality_df = pd.DataFrame({
    "column": df.columns,
    "unique_count": df.nunique(dropna=True).values,
    "unique_%": df.nunique(dropna=True).values / len(df) * 100,
    "dtype": df.dtypes.astype(str).values
}).sort_values("unique_count", ascending=False)

display(cardinality_df)


section("Cardinality groups")

cardinality_groups = pd.DataFrame({
    "group": [
        "High-cardinality columns, more than 90% unique",
        "Low-cardinality columns, 10 or fewer unique values"
    ],
    "columns": [
        cardinality_df[cardinality_df["unique_%"] > 90]["column"].tolist(),
        cardinality_df[cardinality_df["unique_count"] <= 10]["column"].tolist()
    ]
})

display(cardinality_groups)


section("Constant and near-constant columns")

constant_cols = cardinality_df[cardinality_df["unique_count"] <= 1]["column"].tolist()

near_constant_info = []

for col in df.columns:
    top_freq = df[col].value_counts(dropna=False, normalize=True).head(1)

    if len(top_freq) > 0:
        most_common_pct = top_freq.iloc[0] * 100

        if most_common_pct > 95:
            near_constant_info.append({
                "column": col,
                "most_common_value": top_freq.index[0],
                "most_common_%": most_common_pct
            })

near_constant_df = pd.DataFrame(near_constant_info)

constant_summary = pd.DataFrame({
    "type": ["Constant columns"],
    "columns": [constant_cols]
})

display(constant_summary)

if len(near_constant_df) > 0:
    display(near_constant_df.sort_values("most_common_%", ascending=False))
else:
    display(pd.DataFrame({"result": ["No near-constant columns detected"]}))


section("Numeric columns")

numeric_cols = df.select_dtypes(include=np.number).columns

numeric_summary = df[numeric_cols].describe().T
display(numeric_summary)


section("Numeric outliers using IQR method")

outlier_info = []

for col in numeric_cols:
    series = df[col].dropna()

    if len(series) == 0:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    if iqr == 0:
        continue

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = ((series < lower_bound) | (series > upper_bound)).sum()

    outlier_info.append({
        "column": col,
        "outlier_count": outliers,
        "outlier_%": outliers / len(series) * 100,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "min": series.min(),
        "max": series.max()
    })

outlier_df = pd.DataFrame(outlier_info)

if len(outlier_df) > 0:
    display(outlier_df.sort_values("outlier_%", ascending=False))
else:
    display(pd.DataFrame({"result": ["No numeric outliers detected"]}))


section("Object and categorical columns")

categorical_summary = []

for col in object_cols:
    value_counts = df[col].value_counts(dropna=False)
    top_value = value_counts.index[0] if len(value_counts) > 0 else None
    top_count = value_counts.iloc[0] if len(value_counts) > 0 else 0
    top_pct = top_count / len(df) * 100

    categorical_summary.append({
        "column": col,
        "unique_count": df[col].nunique(dropna=True),
        "missing_%": df[col].isnull().mean() * 100,
        "top_value": top_value,
        "top_value_%": top_pct
    })

categorical_summary_df = pd.DataFrame(categorical_summary)
categorical_summary_df = categorical_summary_df.sort_values("unique_count", ascending=False)

display(categorical_summary_df)


section("Top values for low-cardinality columns")

low_card_cols = cardinality_df[
    (cardinality_df["unique_count"] <= 20) &
    (cardinality_df["unique_count"] > 1)
]["column"].tolist()

top_values_tables = []

for col in low_card_cols:
    temp = df[col].value_counts(dropna=False).head(10).reset_index()
    temp.columns = ["value", "count"]
    temp.insert(0, "column", col)
    top_values_tables.append(temp)

if len(top_values_tables) > 0:
    top_values_df = pd.concat(top_values_tables, ignore_index=True)
    display(top_values_df)
else:
    display(pd.DataFrame({"result": ["No low-cardinality columns found"]}))


section("Date columns check")

possible_date_cols = [
    col for col in df.columns
    if "date" in col.lower() or "time" in col.lower()
]

date_info = []

for col in possible_date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce")
    invalid_count = parsed.isnull().sum() - df[col].isnull().sum()

    date_info.append({
        "column": col,
        "current_dtype": df[col].dtype,
        "missing_before_parsing": df[col].isnull().sum(),
        "invalid_dates_after_parsing": invalid_count,
        "min_date": parsed.min() if parsed.notnull().sum() > 0 else None,
        "max_date": parsed.max() if parsed.notnull().sum() > 0 else None
    })

date_info_df = pd.DataFrame(date_info)
display(date_info_df)


section("Created and closed date consistency")

if "Created Date" in df.columns and "Closed Date" in df.columns:
    created = pd.to_datetime(df["Created Date"], errors="coerce")
    closed = pd.to_datetime(df["Closed Date"], errors="coerce")

    resolution_time_hours = (closed - created).dt.total_seconds() / 3600

    consistency_df = pd.DataFrame({
        "metric": [
            "Invalid Created Date values",
            "Invalid Closed Date values",
            "Missing Closed Date values",
            "Rows where Closed Date is before Created Date",
            "Negative resolution times",
            "Resolution times greater than 1 year"
        ],
        "value": [
            created.isnull().sum() - df["Created Date"].isnull().sum(),
            closed.isnull().sum() - df["Closed Date"].isnull().sum(),
            closed.isnull().sum(),
            (closed < created).sum(),
            (resolution_time_hours < 0).sum(),
            (resolution_time_hours > 24 * 365).sum()
        ]
    })

    display(consistency_df)

    resolution_summary = resolution_time_hours.describe().reset_index()
    resolution_summary.columns = ["statistic", "resolution_time_hours"]

    display(resolution_summary)


section("Geographic columns check")

geo_cols = ["Latitude", "Longitude", "Location", "Incident Zip", "BBL"]

geo_info = []

for col in geo_cols:
    if col in df.columns:
        row = {
            "column": col,
            "dtype": df[col].dtype,
            "missing_%": round(df[col].isnull().mean() * 100, 2),
            "unique_values": df[col].nunique(dropna=True),
            "non_numeric_after_conversion": None,
            "min_after_conversion": None,
            "max_after_conversion": None
        }

        if col in ["Latitude", "Longitude"]:
            converted = pd.to_numeric(
                df[col].astype(str).str.replace(",", ".", regex=False),
                errors="coerce"
            )

            row["non_numeric_after_conversion"] = converted.isnull().sum() - df[col].isnull().sum()
            row["min_after_conversion"] = converted.min()
            row["max_after_conversion"] = converted.max()

        geo_info.append(row)

geo_info_df = pd.DataFrame(geo_info)
display(geo_info_df)


section("Potential data leakage columns")

leakage_keywords = ["closed", "resolution", "status", "due"]

potential_leakage_cols = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

leakage_df = pd.DataFrame({
    "potential_leakage_column": potential_leakage_cols
})

display(leakage_df)


section("Final diagnostic summary")

final_summary = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Object columns",
        "Numeric columns",
        "Columns with more than 50% missing",
        "Columns with 10% to 50% missing",
        "High-cardinality columns",
        "Constant columns",
        "Fully duplicated rows",
        "Rows with missing Closed Date"
    ],
    "value": [
        n_rows,
        n_cols,
        len(object_cols),
        len(numeric_cols),
        len(missing_df[missing_df["missing_%"] > 50]),
        len(missing_df[(missing_df["missing_%"] >= 10) & (missing_df["missing_%"] <= 50)]),
        len(cardinality_df[cardinality_df["unique_%"] > 90]),
        len(constant_cols),
        df.duplicated().sum(),
        df["Closed Date"].isnull().sum() if "Closed Date" in df.columns else None
    ]
})

display(final_summary)

issues = pd.DataFrame({
    "main_issue": [
        "Many columns are stored as object dtype.",
        "Some columns require date parsing.",
        "Some columns have high missingness.",
        "Some columns may be ID-like or too high-cardinality for direct modelling.",
        "Geographic columns may require type conversion or parsing.",
        "Closed Date may create missing target values or data leakage depending on the modelling objective."
    ]
})

display(issues)

## Dataset overview

,metric,value
0,Rows,110930.0
1,Columns,41.0
2,Total cells,4548130.0
3,Memory usage MB,234.7


## Column types

,dtype,count
0,object,36
1,float64,3
2,int64,2


,column,dtype,non_null_count,null_count,missing_%,unique_count,unique_%
0,Unnamed: 0,int64,110930,0,0.000000,110930,100.000000
1,Unique Key,int64,110930,0,0.000000,110930,100.000000
2,Created Date,object,110930,0,0.000000,95296,85.906427
3,Closed Date,object,91738,19192,17.301001,73517,66.273326
4,Agency,object,110930,0,0.000000,15,0.013522
5,Agency Name,object,110930,0,0.000000,15,0.013522
6,Problem (formerly Complaint Type),object,110930,0,0.000000,163,0.146940
7,Problem Detail (formerly Descriptor),object,109098,1832,1.651492,700,0.631029
8,Additional Details,object,41038,69892,63.005499,577,0.520148
9,Location Type,object,93679,17251,15.551249,106,0.095556


...

## Potential data leakage columns

,potential_leakage_column
0,Closed Date


## Final diagnostic summary

,metric,value
0,Rows,110930
1,Columns,41
2,Object columns,36
3,Numeric columns,5
4,Columns with more than 50% missing,9
5,Columns with 10% to 50% missing,8
6,High-cardinality columns,2
7,Constant columns,1
8,Fully duplicated rows,0
9,Rows with missing Closed Date,19192


,main_issue
0,Many columns are stored as object dtype.
1,Some columns require date parsing.
2,Some columns have high missingness.
3,Some columns may be ID-like or too high-cardinality for direct modelling.
4,Geographic columns may require type conversion or parsing.
5,Closed Date may create missing target values or data leakage depending on the modelling objective.
